In [2]:
import numpy as np
import random
import pandas as pd
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

### Funções Derivation Evolution

In [12]:
# ---------------------------------------------------------------
# FUNÇÕES OBJETIVO
# ---------------------------------------------------------------
# 1: Função Esférica
def sphere(x):  # [ALTERADO] só limpei os espaços (np. sum -> np.sum) e a anotação de tipo
    return np.sum(x**2)


# 2: Função Ackley (sem alteração)
def ackley(x):
    n = len(x)
    return (-20*np.exp(-0.2*np.sqrt(np.sum(x**2)/n))
            - np.exp(np.mean(np.cos(2*np.pi*x))) + 20 + np.e)


# 3: Função Rastrigin (sem alteração)
def rastrigin(x):
    n = len(x)
    return 10*n + np.sum(x**2 - 10*np.cos(2*np.pi*x))


# [NOVO] dicionário para chamar a função objetivo pelo nome (string)
FUNCOES = {'esfera': sphere, 'ackley': ackley, 'rastrigin': rastrigin}

# [NOVO] limites de caixa usuais de cada função (o enunciado não os informa:
# justifique essa escolha no relatório)
LIMITES = {'esfera': [-5.12, 5.12],
           'ackley': [-32.768, 32.768],
           'rastrigin': [-5.12, 5.12]}


# ---------------------------------------------------------------
# POPULAÇÃO INICIAL (sem alteração)
# ---------------------------------------------------------------
def cria_individuos(n, limites, rng):
    """Sorteia um indivíduo uniforme dentro dos limites. rng garante reprodutibilidade."""
    individuo = rng.uniform(limites[0], limites[1], n)
    return individuo


def criar_populacao(tamanho, n, limites, rng):  # gera 'tamanho' indivíduos
    populacao = []
    for i in range(tamanho):
        populacao.append(cria_individuos(n, limites, rng))
    populacao = np.array(populacao)
    return populacao


# ---------------------------------------------------------------
# OPERADORES DO DE
# ---------------------------------------------------------------
def escolher(populacao, i, rng, x=3):
    # [ALTERADO] comentário e docstring corrigidos
    """Sorteia x indivíduos distintos entre si e diferentes do target i.
    Retorna os vetores r1, r2, r3 usados na mutação."""
    fim = len(populacao)
    valores_possiveis = np.arange(0, fim)
    valores_filtrados = valores_possiveis[valores_possiveis != i]  # exclui o target
    indices = rng.choice(valores_filtrados, size=x, replace=False)  # [ALTERADO] rng.choice (antes rng.random.choice)
    sorteados = populacao[indices]
    return sorteados


def mutacao(sorteados, f):
    """DE/rand/1: v = r1 + F*(r2 - r3). (sem alteração)"""
    vi = sorteados[0] + (f * (sorteados[1] - sorteados[2]))
    return vi


def crossover(target, vi, cr, rng):
    # [ALTERADO] reescrita: recebe o target como vetor 1D (populacao[i]) e devolve trial 1D
    """Crossover binomial. cr = probabilidade de copiar a coordenada do mutante.
    Pelo menos uma coordenada (j_rand) sempre vem do mutante."""
    n = len(vi)
    # [REMOVIDO] target = target[0]  -> o target agora já é um vetor 1D
    # [ALTERADO] rng.random(n) em vez de rng.random((1, n)): usa pb[j] em vez de pb[0][j]
    pb = rng.random(n)
    mascara = pb < cr                      # [ALTERADO] versão vetorizada, substitui o for/append
    mascara[rng.integers(0, n)] = True     # [ALTERADO] j_rand com rng.integers (antes random.randint)
    trial = np.where(mascara, vi, target)  # mutante onde mascara=True, target onde False
    # [REMOVIDO] np.reshape(trial, (1, -1)) -> trial permanece 1D
    return trial


# ---------------------------------------------------------------
# RESTRIÇÕES E PENALIZAÇÃO (dadas pelo professor, sem alteração)
# ---------------------------------------------------------------
def restricoes(x, funcao):
    g1 = np.sum(x) - 5
    if funcao == 'esfera':
        g2 = np.sum(x**2) - 25
    elif funcao == 'ackley':
        g2 = np.sum(x**2) - 50
    elif funcao == 'rastrigin':
        g2 = np.sum(np.cos(x)) - 1
    else:
        raise ValueError('Função desconhecida')
    return np.array([g1, g2])


def valor_penalizado(x, funcao, rho):
    f = {'esfera': sphere, 'ackley': ackley, 'rastrigin': rastrigin}[funcao](x)
    g = restricoes(x, funcao)
    return f + rho*np.sum(np.maximum(0, g)**2)


# [NOVO] avalia UM indivíduo e devolve (valor penalizado, f original, é viável?)
# Calcula f e g uma única vez. É aqui que restrições e penalização entram no DE.
def avaliar_individuo(x, funcao, rho):
    f = FUNCOES[funcao](x)
    g = restricoes(x, funcao)
    f_pen = f + rho*np.sum(np.maximum(0, g)**2)
    viavel = bool(np.all(g <= 0))
    return f_pen, f, viavel


# (sem alteração, mas NÃO é mais usada no de: avaliamos um indivíduo por vez
# para poder contar NFE e checar viabilidade a cada avaliação)
def avaliar(funcao, populacao, rho):
    fitness = np.apply_along_axis(lambda x: valor_penalizado(x, funcao, rho),
                                  axis=1, arr=populacao)
    return fitness


# ---------------------------------------------------------------
# SELEÇÃO
# ---------------------------------------------------------------
def torneio(fit_target, target, fit_trial, trial):
    # [ALTERADO] recebe só os dois competidores e devolve (vetor, fitness) do vencedor,
    # para o de poder guardar o fitness sem reavaliar. Empate favorece o trial (<=).
    if fit_trial <= fit_target:
        return trial, fit_trial
    else:
        return target, fit_target


### ALGORITMMO DE

In [13]:
# ---------------------------------------------------------------
# ALGORITMO DE
# ---------------------------------------------------------------
# [ALTERADO] assinatura: f_o (função) virou 'funcao' (string); nfe_inicio removido (sempre 0)
def de(seed, tamanho, n,  f, cr, funcao, rho, nfe_max, limites):
    rng = np.random.default_rng(seed)
    lo, hi = limites                         # [NOVO] limites de caixa
    historico_convergencia = []              # [ALTERADO] guardará pares (nfe, melhor penalizado)

    populacao = criar_populacao(tamanho, n, limites, rng)

    # [NOVO] melhor solução VIÁVEL (com f original). None = ainda não achou viável
    melhor_x, melhor_f = None, np.inf

    # [NOVO] avaliação da população inicial UMA vez, contando NFE
    fitness = np.empty(tamanho)
    nfe = 0
    for k in range(tamanho):
        fitness[k], f_orig, viavel = avaliar_individuo(populacao[k], funcao, rho)
        nfe += 1
        if viavel and f_orig < melhor_f:
            melhor_x, melhor_f = populacao[k].copy(), f_orig
    historico_convergencia.append((nfe, fitness.min()))

    # [ALTERADO] critério de parada só por NFE (antes: "or fitness_min < 0", que dava NameError)
    while nfe < nfe_max:
        nova_populacao = populacao.copy()    # [ALTERADO] cópia; vencedores entram no lugar do target
        novo_fitness = fitness.copy()        # [NOVO] fitness acompanha a população

        # [ALTERADO] for com i = índice do target (antes: while com target aleatório)
        for i in range(tamanho):
            if nfe >= nfe_max:               # [NOVO] garante que NFE nunca passa do orçamento
                break

            target = populacao[i]            # [ALTERADO] target é o indivíduo i (escolher_target removida)
            escolhidos = escolher(populacao, i, rng, 3)   # [ALTERADO] argumentos (populacao, i, rng)
            vi = mutacao(escolhidos, f)
            trial = crossover(target, vi, cr, rng)
            trial = np.clip(trial, lo, hi)   # [NOVO] limites de caixa (projeção), ANTES de avaliar

            # [ALTERADO] avalia só o trial (antes reavaliava a população inteira a cada vez)
            f_pen_trial, f_orig, viavel = avaliar_individuo(trial, funcao, rho)
            nfe += 1                         # [ALTERADO] +1 por avaliação

            # [NOVO] registro do melhor viável (compara o f ORIGINAL, não o penalizado)
            if viavel and f_orig < melhor_f:
                melhor_x, melhor_f = trial.copy(), f_orig

            # [ALTERADO] seleção usa o fitness já guardado do target (fitness[i])
            vencedor, fit_vencedor = torneio(fitness[i], target, f_pen_trial, trial)
            nova_populacao[i] = vencedor     # [ALTERADO] sem "vencedor[0]": agora é vetor 1D
            novo_fitness[i] = fit_vencedor

        populacao, fitness = nova_populacao, novo_fitness
        # [REMOVIDO] avaliação extra da nova população no fim da geração
        historico_convergencia.append((nfe, fitness.min()))  # [ALTERADO] guarda também o NFE
        # [REMOVIDO] print(nfe) dentro do laço

    # [ALTERADO] retorno: melhor viável (None se não houve), f original, histórico e NFE
    return populacao, melhor_x, melhor_f if melhor_x is not None else None, \
        historico_convergencia, nfe

### Testes das funções

2. criar_populacao(tamanho, n, limites, rng): # retorna uma população de tamanho == tamanho
    1. cria_individuos (n, limites, rng): # retorna um individuo
3. escolher(populacao, rng, x=3): # retorna tres individuos diferentes: a, b e c
4. mutacao (sorteados, f): # retorna um array 1d
5. escolher_target(populacao, sorteados): # retorna um array com o target e um com a posição
6. crossover(target, vi, cr): # retorna o trial
8. avaliar(funcao_objetivo, populacao): # retorna um array com o tamanho da população
    7. sphere (x: np. ndarray ) -> float: # retorna o fitnees individual
9. torneio(fitness, posicao, população, trail, fitness_vetor_mutante, rng): # retorna um array e uma posição

In [5]:
seed = 20
tamanho = 10
n = 2
# limites = [-5.12, 5.12]
f = 0.5
cr = 0.5
funcao = "esfera"
nfe_inicio = 0
nfe_max = 2000
rho = 100


In [6]:
# [NOVO] teste rápido (só roda se executar este arquivo diretamente)
if __name__ == "__main__":
    # funcao, n = 'esfera', 2
    t0 = time.time()
    f= 0.5
    pop, bx, bf, hist, nfe = de(seed, tamanho, n, f, cr,
                        funcao, rho, nfe_max, limites=LIMITES[funcao])
    print("NFE final:", nfe)
    print("Melhor viável:", bx, "| f =", bf)
    print("Tempo (s):", round(time.time() - t0, 3))

NFE final: 2000
Melhor viável: [ 1.12787243e-23 -9.90496902e-23] | f = 9.938050750281136e-45
Tempo (s): 0.052


In [7]:
# [NOVO] teste rápido (só roda se executar este arquivo diretamente)
if __name__ == "__main__":
    funcao, n = 'esfera', 2
    t0 = time.time()
    pop, bx, bf, hist, nfe = de(seed=42, tamanho=20, n=n, limites=LIMITES[funcao],
                                f=0.5, cr=0.9, funcao=funcao, rho=1, nfe_max=20000)
    print("NFE final:", nfe)
    print("Melhor viável:", bx, "| f =", bf)
    print("Tempo (s):", round(time.time() - t0, 3))

NFE final: 20000
Melhor viável: [ 1.23734962e-110 -4.08223772e-111] | f = 1.6976807371407854e-220
Tempo (s): 0.38


# Rodar experimentos

In [14]:
"""
Experimentos do Trabalho Prático 2 - Differential Evolution (DE/rand/1/Bin)

Coloque este arquivo na mesma pasta de de_corrigido.py e execute:
    python experimentos.py

Gera:
    resultados_principais.csv   -> tabela com as métricas da seção 6
    resultados_parametro.csv    -> experimento da seção 5 (Ackley)
    resultados_rho.csv          -> estudo da influência de rho (extra, para a discussão)
    melhores_solucoes.txt       -> melhor vetor viável de cada função/dimensão
    convergencia_<funcao>.png   -> gráficos de convergência (valor penalizado x NFE)
    convergencia_parametro.png  -> convergência das duas configurações comparadas
"""


# from de_corrigido import de, LIMITES

# ---------------------------------------------------------------
# CONFIGURAÇÃO (seção 4 do enunciado)
# ---------------------------------------------------------------
SEMENTES = list(range(10))        # 10 execuções independentes; as sementes ficam registradas
NFE_MAX = 20000                   # orçamento por execução
NOMES_FUNCOES = ['esfera', 'ackley', 'rastrigin'] #FUNCOES = ['esfera', 'ackley', 'rastrigin']
DIMENSOES = [2, 5, 10]

# Configuração básica do DE (a mesma para todas as funções)
NP = 50
F = 0.5
CR = 0.9
# rho pode variar por função, mas fica fixo dentro de cada função
RHO = {'esfera': 100, 'ackley': 100, 'rastrigin': 100}

# Experimento da seção 5: UMA dimensão de Ackley, duas configs que diferem em UM fator
DIM_PARAM = 10
FATOR = 'F'                                   # 'F', 'CR' ou 'NP'
CONFIG_A = dict(NP=NP, F=0.5, CR=CR)          # base
CONFIG_B = dict(NP=NP, F=0.9, CR=CR)          # só o F muda

# Estudo extra da influência de rho (função e dimensão fixas)
RHOS_TESTE = [1, 100, 10000]
FUNCAO_RHO, DIM_RHO = 'rastrigin', 10


# ---------------------------------------------------------------
# FUNÇÕES AUXILIARES
# ---------------------------------------------------------------
def executar(funcao, n, rho, cfg):
    """Roda as 10 sementes de uma configuração. Devolve lista de dicionários."""
    runs = []
    for s in SEMENTES:
        t0 = time.time()
        _, bx, bf, hist, nfe = de(seed=s, tamanho=cfg['NP'], n=n, limites=LIMITES[funcao],
                                  f=cfg['F'], cr=cfg['CR'], funcao=funcao, rho=rho,
                                  nfe_max=NFE_MAX)
        runs.append(dict(seed=s, x=bx, f=bf, hist=np.array(hist),
                         nfe=nfe, tempo=time.time() - t0))
    return runs


def resumir(runs):
    """Métricas da seção 6 para um conjunto de execuções."""
    viaveis = [r for r in runs if r['x'] is not None]
    fs = np.array([r['f'] for r in viaveis])
    resumo = dict(
        n_viaveis=len(viaveis),
        taxa_sucesso=len(viaveis) / len(runs),
        media=fs.mean() if len(fs) else np.nan,
        desvio=fs.std(ddof=1) if len(fs) > 1 else np.nan,
        melhor=fs.min() if len(fs) else np.nan,
        nfe_medio=np.mean([r['nfe'] for r in runs]),
        tempo_medio=np.mean([r['tempo'] for r in runs]),
    )
    if len(viaveis):
        resumo['melhor_x'] = viaveis[int(np.argmin(fs))]['x']
    else:
        resumo['melhor_x'] = None   # nenhuma solução viável: registrar, sem usar valor penalizado
    return resumo


def curva_media(runs):
    """Curva média (e NFE) do melhor valor PENALIZADO; todas as sementes compartilham o mesmo NFE."""
    nfes = runs[0]['hist'][:, 0]
    vals = np.vstack([r['hist'][:, 1] for r in runs])
    return nfes, vals.mean(axis=0), vals


def plotar(ax, nfes, media, vals, rotulo):
    ax.plot(nfes, np.maximum(media, 1e-20), label=rotulo)
    ax.fill_between(nfes, np.maximum(vals.min(axis=0), 1e-20),
                    np.maximum(vals.max(axis=0), 1e-20), alpha=0.15)




In [16]:
# ---------------------------------------------------------------
# 1) EXPERIMENTO PRINCIPAL: 3 funções x 3 dimensões x 10 sementes
# ---------------------------------------------------------------


print("=== Experimento principal ===")
linhas, melhores_txt = [], []
for funcao in NOMES_FUNCOES:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for n in DIMENSOES:
        runs = executar(funcao, n, RHO[funcao], dict(NP=NP, F=F, CR=CR))
        r = resumir(runs)
        linhas.append(dict(funcao=funcao, n=n, rho=RHO[funcao], NP=NP, F=F, CR=CR,
                           n_viaveis=r['n_viaveis'], taxa_sucesso=r['taxa_sucesso'],
                           media_f=r['media'], desvio_f=r['desvio'], melhor_f=r['melhor'],
                           nfe_medio=r['nfe_medio'], tempo_medio_s=r['tempo_medio']))
        melhores_txt.append(f"{funcao} n={n}: f={r['melhor']}  x={r['melhor_x']}")
        print(f"{funcao:10s} n={n:2d} viáveis={r['n_viaveis']}/10  "
              f"média={r['media']:.4g}  dp={r['desvio']:.4g}  melhor={r['melhor']:.4g}")
        nfes, media, vals = curva_media(runs)
        plotar(ax, nfes, media, vals, f"n={n}")
    ax.set_yscale('log')
    ax.set_xlabel("NFE")
    ax.set_ylabel("Melhor valor penalizado (média das 10 execuções)")
    ax.set_title(f"Convergência - {funcao} (rho={RHO[funcao]})")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"convergencia_{funcao}.png", dpi=150)
    plt.close(fig)

pd.DataFrame(linhas).to_csv("resultados_principais.csv", index=False)
with open("melhores_solucoes.txt", "w", encoding="utf-8") as fh:
    fh.write("\n".join(melhores_txt))

# ---------------------------------------------------------------
# 2) EXPERIMENTO DE PARÂMETRO (seção 5): Ackley, uma dimensão, um fator
# ---------------------------------------------------------------
print(f"\n=== Parâmetro {FATOR} em Ackley n={DIM_PARAM} ===")
linhas_p = []
fig, ax = plt.subplots(figsize=(7, 4.5))
for nome, cfg in [("A", CONFIG_A), ("B", CONFIG_B)]:
    runs = executar('ackley', DIM_PARAM, RHO['ackley'], cfg)   # mesmas sementes e mesmo NFE
    r = resumir(runs)
    linhas_p.append(dict(config=nome, **cfg, rho=RHO['ackley'], n_viaveis=r['n_viaveis'],
                         taxa_sucesso=r['taxa_sucesso'], media_f=r['media'],
                         desvio_f=r['desvio'], melhor_f=r['melhor'],
                         nfe_medio=r['nfe_medio'], tempo_medio_s=r['tempo_medio']))
    print(f"Config {nome} {cfg}: média={r['media']:.4g}  dp={r['desvio']:.4g}  melhor={r['melhor']:.4g}")
    nfes, media, vals = curva_media(runs)
    plotar(ax, nfes, media, vals, f"Config {nome}: {FATOR}={cfg[FATOR]}")
ax.set_yscale('log')
ax.set_xlabel("NFE")
ax.set_ylabel("Melhor valor penalizado (média das 10 execuções)")
ax.set_title(f"Ackley n={DIM_PARAM}: efeito de {FATOR}")
ax.legend()
fig.tight_layout()
fig.savefig("convergencia_parametro.png", dpi=150)
plt.close(fig)
pd.DataFrame(linhas_p).to_csv("resultados_parametro.csv", index=False)

# ---------------------------------------------------------------
# 3) ESTUDO DE rho (apoia a discussão "influência de rho")
# ---------------------------------------------------------------
print(f"\n=== Influência de rho em {FUNCAO_RHO} n={DIM_RHO} ===")
print(f"\n=== Influência de rho em {FUNCAO_RHO} n={DIM_RHO} ===")
linhas_r = []
for rho in RHOS_TESTE:
    runs = executar(FUNCAO_RHO, DIM_RHO, rho, dict(NP=NP, F=F, CR=CR))
    r = resumir(runs)
    linhas_r.append(dict(funcao=FUNCAO_RHO, n=DIM_RHO, rho=rho, n_viaveis=r['n_viaveis'],
                         taxa_sucesso=r['taxa_sucesso'], media_f=r['media'],
                         desvio_f=r['desvio'], melhor_f=r['melhor']))
    print(f"rho={rho:>6}: viáveis={r['n_viaveis']}/10  média={r['media']:.4g}  melhor={r['melhor']:.4g}")
pd.DataFrame(linhas_r).to_csv("resultados_rho.csv", index=False)

print("\nArquivos gerados. Sementes usadas:", SEMENTES)

=== Experimento principal ===
esfera     n= 2 viáveis=10/10  média=3.205e-86  dp=1.006e-85  melhor=3.645e-91
esfera     n= 5 viáveis=10/10  média=2.927e-34  dp=3.159e-34  melhor=8.313e-36
esfera     n=10 viáveis=10/10  média=2.956e-16  dp=3.836e-16  melhor=4.232e-17
ackley     n= 2 viáveis=10/10  média=4.441e-16  dp=0  melhor=4.441e-16
ackley     n= 5 viáveis=10/10  média=1.155e-15  dp=1.498e-15  melhor=4.441e-16
ackley     n=10 viáveis=10/10  média=1.475e-07  dp=4.423e-08  melhor=9.469e-08
rastrigin  n= 2 viáveis=10/10  média=3.191  dp=0.08983  melhor=3.098
rastrigin  n= 5 viáveis=10/10  média=12.84  dp=0.9256  melhor=11.31
rastrigin  n=10 viáveis=10/10  média=54.22  dp=4.95  melhor=45.01

=== Parâmetro F em Ackley n=10 ===
Config A {'NP': 50, 'F': 0.5, 'CR': 0.9}: média=1.475e-07  dp=4.423e-08  melhor=9.469e-08
Config B {'NP': 50, 'F': 0.9, 'CR': 0.9}: média=2.666  dp=0.2729  melhor=2.193

=== Influência de rho em rastrigin n=10 ===

=== Influência de rho em rastrigin n=10 ===
rho=  